In [4]:
LAST_NAME = "Rosales"
STUDENT_ID = "TUPM-26-3624"
SEED_DIGIT = 4
ID_SUM = 23
NAME_LENGTH = len(LAST_NAME)


TARGET_OS ="Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR = (SEED_DIGIT % 5) + 1

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Target Operating System:", TARGET_OS)
print("Target Behavior Class:", TARGET_BEHAVIOR)

Student: Rosales
Student ID: TUPM-26-3624
Target Operating System: Android
Target Behavior Class: 5


In [5]:
import csv
import json
import shutil
from pathlib import Path

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

INPUT_FILE = DATA_DIR / "user_behavior_dataset.csv"

print("Dataset found:", INPUT_FILE.exists())
print("Output Folder Ready:", OUTPUT_DIR.exists())
print("Backup Folder Ready:", BACKUP_DIR.exists())

Dataset found: True
Output Folder Ready: True
Backup Folder Ready: True


In [6]:
with open(INPUT_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    
    print("Column Name:")
    print(reader.fieldnames)

    print("\nFirst 5 Records:")
    for i, row in enumerate(reader):
        if i==5:
            break
        print(row)

Column Name:
['User ID', 'Device Model', 'Operating System', 'App Usage Time (min/day)', 'Screen On Time (hours/day)', 'Battery Drain (mAh/day)', 'Number of Apps Installed', 'Data Usage (MB/day)', 'Age', 'Gender', 'User Behavior Class']

First 5 Records:
{'User ID': '1', 'Device Model': 'Google Pixel 5', 'Operating System': 'Android', 'App Usage Time (min/day)': '393', 'Screen On Time (hours/day)': '6.4', 'Battery Drain (mAh/day)': '1872', 'Number of Apps Installed': '67', 'Data Usage (MB/day)': '1122', 'Age': '40', 'Gender': 'Male', 'User Behavior Class': '4'}
{'User ID': '2', 'Device Model': 'OnePlus 9', 'Operating System': 'Android', 'App Usage Time (min/day)': '268', 'Screen On Time (hours/day)': '4.7', 'Battery Drain (mAh/day)': '1331', 'Number of Apps Installed': '42', 'Data Usage (MB/day)': '944', 'Age': '47', 'Gender': 'Female', 'User Behavior Class': '3'}
{'User ID': '3', 'Device Model': 'Xiaomi Mi 11', 'Operating System': 'Android', 'App Usage Time (min/day)': '154', 'Screen 

In [7]:
selected_users = []
total_records = 0
os_matches = 0

with open(INPUT_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records += 1
        
        if row["Operating System"] == TARGET_OS:
            os_matches += 1

            if int(row["User Behavior Class"]) == TARGET_BEHAVIOR:
                selected_users.append(row)

print("Total Records:", total_records)
print("Records Matching OS:", os_matches)
print("Records Matching Target Behavior:", len(selected_users))

Total Records: 700
Records Matching OS: 554
Records Matching Target Behavior: 105


In [8]:
if len(selected_users) > 0:
    total_battery_drain = sum(
        float(row["Battery Drain (mAh/day)"])
        for row in selected_users
    )

    total_data_usage = sum(
        float(row["Data Usage (MB/day)"])
        for row in selected_users
    )

    average_battery_drain = total_battery_drain / len(selected_users)
    average_data_usage = total_data_usage / len(selected_users)

else:
    average_battery_drain = 0
    average_data_usage = 0

print("Average Battery Drain:", round(average_battery_drain, 2), "mAh/day")
print("Average Data usage:", round(average_data_usage, 2), "MB/day")

Average Battery Drain: 2693.06 mAh/day
Average Data usage: 1984.77 MB/day


In [12]:
EXPORT_FILE = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_usage.csv"

export_fields = [
    "User ID",
    "Device Model", 
    "Operating System",
    "Battery Drain (mAh/day)",\
    "Data Usage (MB/day)"
]

with open(EXPORT_FILE, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=export_fields)
    writer.writeheader()
    writer.writerows(
        {field: row[field] for field in export_fields}
        for row in selected_users
    )

print("Exported CSV:", EXPORT_FILE)
print("Selected users exported:", len(selected_users))

Exported CSV: output\Rosales_TUPM-26-3624_usage.csv
Selected users exported: 105


In [13]:
with open(EXPORT_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    exported_rows = sum(1 for row in reader)

count_verified = exported_rows == len(selected_users)

print("Selected user count:", len(selected_users))
print("Exported CSV row count:", exported_rows)
print("Count verified:", count_verified)

Selected user count: 105
Exported CSV row count: 105
Count verified: True


In [ ]:
SUMMARY_FILE = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_usage_summary.json"

summary = {
    "Student": LAST_NAME,
    "Student ID": STUDENT_ID,
    "Operating System": TARGET_OS,
    "Target Behavior Class": TARGET_BEHAVIOR,
    "Total Records": total_records,
    "Records Matching OS": os_matches,
    "Records Matching Target Behavior": len(selected_users),
    "Average Battery Drain (mAh/day)": round(average_battery_drain, 2),
    "Average Data Usage (MB/day)": round(average_data_usage, 2),
    "Exported CSV Row Count": exported_rows,
    "Count Verified": count_verified
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", SUMMARY_FILE)
print("Summary saved:", SUMMARY_FILE.exists())

JSON summary created: output\Rosales_TUPM-26-3624_summary.json
Summary saved: True


In [15]:
BACKUP_FILE = BACKUP_DIR / EXPORT_FILE.name

shutil.copy2(EXPORT_FILE, BACKUP_FILE)

file_size = BACKUP_FILE.stat().st_size

print("Backup created:", BACKUP_FILE)
print("Backup exists:", BACKUP_FILE.exists())
print("Backup file size:", file_size, "bytes")

Backup created: backup\Rosales_TUPM-26-3624_usage.csv
Backup exists: True
Backup file size: 3959 bytes


In [16]:
# Add backup information to the JSON summary
summary["Backup File"] = BACKUP_FILE.name
summary["Backup File Size (bytes)"] = file_size

with open(SUMMARY_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("Final Verification")
print("-------------------")
print("Selected users:", len(selected_users))
print("Exported CSV rows:", exported_rows)
print("Count verified:", count_verified)
print("CSV exists:", EXPORT_FILE.exists())
print("JSON summary exists:", SUMMARY_FILE.exists())
print("Backup exists:", BACKUP_FILE.exists())
print("Backup size:", file_size, "bytes")

Final Verification
-------------------
Selected users: 105
Exported CSV rows: 105
Count verified: True
CSV exists: True
JSON summary exists: True
Backup exists: True
Backup size: 3959 bytes
